# 45 DQA-FedSTO-MoX Long 12h

mAP0.6を狙うために、短期judgerではなく長めの二段階学習を行う。既存の50 epoch warmupを読み込み、12時間枠に収めるため `18 + 12 = 30` FL rounds に圧縮する。

## 設計

- Phase 1: selective training。`backbone_moe_head` を低LRで動かし、pseudo labelで表現/routerをtargetへ寄せる。
- Phase 2: full + orthogonal training。`all` をさらに低LRで動かし、head/box/objectness/classを実際に育てる。
- local EMA pseudo labelerを使う。
- DQAでpseudo quality / class coverage / stabilityをゲートする。
- hybrid DQA routerでdomain/class expertを専門化する。
- 21の結果を踏まえ、DQA softmixはBN/MoEを弱く入れ、server anchorを残して崩壊を抑える。
- Discordには開始、進捗、終了を通知する。

In [ ]:
from pathlib import Path
ROOT = Path('/app/Object_Detection')
SCRIPT = ROOT / 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/scripts/run_45_dqa_fedsto_mox_long_12h.py'
WORKSPACE = ROOT / 'dynamic_quality_aware_classwise_aggregation/scene_daynight_dqa/aggressive_dqamox/output/45_dqa_fedsto_mox_long_12h'
print(SCRIPT)
print(WORKSPACE)

In [ ]:
import subprocess, sys
cmd = [
    sys.executable, str(SCRIPT),
    '--workspace-root', str(WORKSPACE),
    '--phase1-rounds', '18',
    '--phase2-rounds', '12',
    '--client-limit', '2200',
    '--imgsz', '640',
    '--pseudo-imgsz', '1152',
    '--batch-size', '80',
    '--val-batch-size', '32',
    '--workers', '48',
    '--gpus', '2',
    '--target-map50', '0.60',
]
print(' '.join(cmd))
subprocess.run(cmd, check=True)

In [ ]:
report = WORKSPACE / '45_dqa_fedsto_mox_long_12h_report.md'
manifest = WORKSPACE / '45_dqa_fedsto_mox_long_12h_manifest.json'
metrics = WORKSPACE / 'stats/18_client_balanced_single_injection_dqamox_final_metrics.csv'
for path in [manifest, metrics, report]:
    print('\n###', path)
    if path.exists():
        print(path.read_text(encoding='utf-8')[:5000])
    else:
        print('missing')